# Регрессия CC50

Этот ноутбук запускает подбор моделей для `CC50, mM` и затем показывает, кто выиграл и за счёт чего.

По EDA у CC50 заметнее линейные связи с частью дескрипторов, но деревья тоже стоит проверять.

In [9]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from types import SimpleNamespace

import pandas as pd
from IPython.display import Image, display

from src.common.config import RESULTS_DIR, TASKS
from src.common.data import find_dataset_path, load_dataset
from src.common.preprocessing import prepare_task_data
from src.common.training import run_regression_si_task, run_supervised_task

## Настройки запуска

По умолчанию стоит полный режим `nested`. Если нужен быстрый прогон для проверки,
достаточно переключить `EVALUATION_STRATEGY` на `"holdout"`.

In [3]:
TASK_NAME = 'regression_cc50'
EVALUATION_STRATEGY = 'nested'
MODELS = None
SKIP_CATBOOST = False
OUTER_FOLDS = 5
INNER_FOLDS = 3
TEST_SIZE = 0.2
RANDOM_SEED = 42
TOP_K_IMPORTANCE = 20

task = TASKS[TASK_NAME]
dataframe = load_dataset(find_dataset_path())
prepared = prepare_task_data(dataframe, task)

print(f'Задача: {task.title}')
print(f'Матрица признаков: {prepared["X"].shape}')
print(f'Число признаков после фильтрации: {len(prepared["feature_columns"])}')
print(f'Статус проверки на утечку: {prepared["leakage_report"]["status"]}')

Задача: Регрессия: CC50
Матрица признаков: (1001, 210)
Число признаков после фильтрации: 210
Статус проверки на утечку: passed


## Быстрый срез по таргету

Перед обучением полезно один раз посмотреть, что именно мы подаём в модель.

In [10]:
display(prepared['y'].describe().to_frame(name=task.target_column).T)

,count,mean,std,min,25%,50%,75%,max
"CC50, mM",1001.0,589.110728,642.867508,0.700808,99.999036,411.039342,894.089176,4538.976189


## Запуск эксперимента

Эта ячейка пересчитывает результаты, пишет артефакты в `results/` и обновляет текстовый отчёт в `reports/`.

In [5]:
args = SimpleNamespace(
    evaluation_strategy=EVALUATION_STRATEGY,
    models=MODELS,
    skip_catboost=SKIP_CATBOOST,
    outer_folds=OUTER_FOLDS,
    inner_folds=INNER_FOLDS,
    test_size=TEST_SIZE,
    random_seed=RANDOM_SEED,
    top_k_importance=TOP_K_IMPORTANCE,
)

summary = run_supervised_task(task, args)
summary

2026-04-20 17:53:12,496 | INFO | Running regression_cc50 with models: ['dummy', 'ridge', 'lasso', 'knn', 'svr', 'random_forest', 'extra_trees', 'gradient_boosting', 'catboost']
2026-04-20 17:53:12,496 | INFO | Evaluating model dummy
2026-04-20 17:53:13,240 | INFO | Evaluating model ridge
2026-04-20 17:53:14,354 | INFO | Evaluating model lasso
/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.732e-01, tolerance: 1.358e-01
  model = cd_fast.enet_coordinate_descent(
/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to i

{'task_name': 'regression_cc50',
 'title': 'Регрессия: CC50',
 'problem_type': 'regression',
 'target_column': 'CC50, mM',
 'threshold': None,
 'primary_metric': 'mae',
 'evaluation_strategy': 'nested',
 'random_seed': 42,
 'data_contract_path': '/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry/results/data_contract.json',
 'leaderboard_path': '/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry/results/regression_cc50/leaderboard.csv',
 'winner': {'task_name': 'regression_cc50',
  'problem_type': 'regression',
  'target_column': 'CC50, mM',
  'primary_metric': 'mae',
  'evaluation_strategy': 'nested',
  'model_name': 'extra_trees',
  'mode': 'direct',
  'fit_seconds': 112.0689982919721,
  'best_params_json': '{"max_depth": null, "max_features": 0.5, "min_samples_leaf": 1}',
  'mae': 279.4717970536836,
  'mae_std': 28.193448341490917,
  'rmse': 472.709549436929,
  'rmse_std': 51.881207741747374,
  'r2': 0.44939425686221457,
  '

## Лидерборд

Сравнение разных моделей.

In [6]:
leaderboard = pd.read_csv(RESULTS_DIR / 'regression_cc50' / 'leaderboard.csv')
display(leaderboard)

,task_name,problem_type,target_column,primary_metric,evaluation_strategy,model_name,mode,fit_seconds,best_params_json,mae,mae_std,rmse,rmse_std,r2,r2_std
0,regression_cc50,regression,"CC50, mM",mae,nested,extra_trees,direct,112.068998,"{""max_depth"": null, ""max_features"": 0.5, ""min_...",2.794718e+02,2.819345e+01,4.727095e+02,5.188121e+01,4.493943e-01,7.240652e-02
1,regression_cc50,regression,"CC50, mM",mae,nested,knn,direct,5.886072,"{""n_neighbors"": 5, ""p"": 1, ""weights"": ""distance""}",2.871441e+02,3.210543e+01,4.990464e+02,6.288549e+01,3.833916e-01,1.127874e-01
2,regression_cc50,regression,"CC50, mM",mae,nested,svr,direct,14.471194,"{""C"": 1.0, ""epsilon"": 0.01, ""gamma"": ""scale""}",2.943505e+02,4.124098e+01,4.899676e+02,9.223335e+01,4.084605e-01,1.583449e-01
3,regression_cc50,regression,"CC50, mM",mae,nested,catboost,direct,898.073734,"{""depth"": 6, ""iterations"": 400, ""l2_leaf_reg"":...",2.948822e+02,2.357044e+01,4.803482e+02,4.963647e+01,4.338446e-01,4.637983e-02
4,regression_cc50,regression,"CC50, mM",mae,nested,random_forest,direct,241.701951,"{""max_depth"": null, ""max_features"": 0.5, ""min_...",2.972878e+02,2.164976e+01,4.805316e+02,3.717809e+01,4.312639e-01,4.102804e-02
5,regression_cc50,regression,"CC50, mM",mae,nested,gradient_boosting,direct,237.139478,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",2.977988e+02,2.451134e+01,4.738725e+02,3.757009e+01,4.456199e-01,5.472720e-02
6,regression_cc50,regression,"CC50, mM",mae,nested,dummy,direct,0.741302,"{""strategy"": ""median""}",4.633846e+02,2.981860e+01,6.637711e+02,6.907274e+01,-7.751880e-02,2.459916e-02
7,regression_cc50,regression,"CC50, mM",mae,nested,lasso,direct,15.085367,"{""alpha"": 0.1}",2.690377e+23,5.380753e+23,3.804767e+24,7.609534e+24,-2.335997e+44,4.671994e+44
8,regression_cc50,regression,"CC50, mM",mae,nested,ridge,direct,1.109263,"{""alpha"": 0.1}",1.576849e+26,3.153697e+26,2.230001e+27,4.460002e+27,-8.024652e+49,1.604930e+50


## Короткий разбор результата

In [7]:
winner = leaderboard.iloc[0]
baseline_rows = leaderboard[leaderboard['model_name'] == 'dummy']
baseline = baseline_rows.iloc[0] if not baseline_rows.empty else None

print(
    f"Победитель: {winner['model_name']} "
    f"({winner['mode']}), "
    f"основная метрика {winner['primary_metric']} = {winner[winner['primary_metric']]:.6f}."
)
if baseline is not None:
    print(
        f"Для сравнения dummy даёт {baseline[baseline['primary_metric']]:.6f} по той же метрике."
    )
print(f"Лучшие параметры: {winner['best_params_json']}")

Победитель: extra_trees (direct), основная метрика mae = 279.471797.
Для сравнения dummy даёт 463.384625 по той же метрике.
Лучшие параметры: {"max_depth": null, "max_features": 0.5, "min_samples_leaf": 1}


## Что видно по важности признаков

In [11]:
importance_path = RESULTS_DIR / 'regression_cc50' / 'winner_feature_importance.csv'
if importance_path.exists():
    display(pd.read_csv(importance_path).head(15))

,feature,importance,abs_importance
0,fr_NH2,0.031441,0.031441
1,NHOHCount,0.021558,0.021558
2,fr_allylic_oxid,0.017746,0.017746
3,Kappa3,0.017458,0.017458
4,PEOE_VSA6,0.016230,0.016230
5,BCUT2D_MWLOW,0.015910,0.015910
6,NumHDonors,0.014815,0.014815
7,MolLogP,0.013992,0.013992
8,PEOE_VSA7,0.013739,0.013739
9,Kappa2,0.012941,0.012941


## Итог
TODO: написать почему именно эта модель победила и порассуждать почему в этой задаче именно она показывает хороший результат